# Messy E-Commerce Sales Dataset

## Contents
- [Imports and Loading](#imports-and-loading)
- [Overview](#overview)
  - [Duplicate check](#duplicate-check)
  - [Column check](#column-check)
  - [Rejected records creation](#rejected-records-creation)
  - [Overall insights](#overall-insights)
- [Data Fromatting and Type Conversion](#data-fromatting-and-type-conversion)
  - [`Order_Date` to datetime64[us]](#order_date-to-datetime64us)
  - [`Quantity` to Int64](#quantity-to-int64)
  - [`Price` to Float64](#price-to-float64)
  - [Formatting strings](#formatting-strings)
  - [Conclusion](#conclusion)
- [ROMI Approach](#romi-approach)
  - [Relation](#relation)
  - [Outlier](#outlier)
    - [Investigating `Price`](#investigating-price)
    - [Investigating `Total`](#investigating-total)
  - [Mismatch](#mismatch)
  - [Interpolation](#interpolation)
- [Final Check](#final-check)
- [Export](#export)

## Imports and Loading

In [281]:
import numpy as np
import pandas as pd

In [282]:
import re # for regex matching

In [283]:
df = pd.read_csv("./data/raw.csv")

print(f"Loaded dataset: {df.shape[0]} rows, {df.shape[1]} columns")

Loaded dataset: 103 rows, 11 columns


## Overview

In [284]:
df.head()

,ID,Customer_Name,Order_ID,Order_Date,Product,Category,Quantity,Price,Payment_Method,Status,Total
0,100,Customer_100,ORD-41285,11/22/2024,Blender,Home,3,38,Cash on Delivery,Shipped,114.00
1,101,Customer_101,ORD-35783,7/5/2025,Smartphone,Electronics,2,abd,PayPal,Processing,NaN
2,102,Customer_102,ORD-84355,12/23/2024,Tennis Racket,Sports,1,389.05,PayPal,Delivered,389.05
3,103,Customer_103,ORD-57811,3/19/2025,Science,Books,5,233.92,PayPal,Processing,1169.60
4,104,Customer_104,ORD-93614,10/20/2025,Biography,Books,1,552.51,Cash on Delivery,Processing,552.51


### Duplicate check

In [285]:
# Duplicate check
print("(1) Duplicates:", df.duplicated(subset=["Order_ID"]).sum())
display(df[df.duplicated(subset=["Order_ID"], keep=False)].sort_values(by="ID")) # keep=False to also include the first occurrences
# drops rows that have the exact same Order_ID, regardless of the other columns
df.drop_duplicates(subset=["Order_ID"], inplace=True)
print("(2) Duplicates:", df.duplicated(subset=["Order_ID"]).sum())

(1) Duplicates: 3


,ID,Customer_Name,Order_ID,Order_Date,Product,Category,Quantity,Price,Payment_Method,Status,Total
42,142,Customer_142,ORD-69018,10/30/2025,Shoes,Clothing,5,645.26,Credit Card,Shipped,2258.410
101,142,Customer_142,ORD-69018,10/30/2025,Shoes,Clothing,5,645.26,Credit Card,Shipped,3226.300
102,146,Customer_146,ORD-32755,7/9/2025,Basketball,electronic,2,705.42,Bank Transfer,Processing,1410.840
46,146,Customer_146,ORD-32755,7/9/2025,Basketball,electronic,2,705.42,Bank Transfer,Processing,1410.840
100,175,Customer_175,ORD-56651,2/24/2025,Headphones,Electronics,1,111.36,Credit Card,Processing,77.952
75,175,Customer_175,ORD-56651,2/24/2025,Headphones,Electronics,1,111.36,Credit Card,Processing,111.360


(2) Duplicates: 0


### Column check

In [286]:
# Column check
print("Initial columns:", df.columns.tolist())
df.columns = df.columns.str.strip()
print("Clean columns:", df.columns.tolist())

Initial columns: ['ID', ' Customer_Name', 'Order_ID', 'Order_Date', 'Product', ' Category', 'Quantity', 'Price', 'Payment_Method', 'Status', 'Total']
Clean columns: ['ID', 'Customer_Name', 'Order_ID', 'Order_Date', 'Product', 'Category', 'Quantity', 'Price', 'Payment_Method', 'Status', 'Total']


### Rejected records creation

In [287]:
# initialize an empty dataframe to store rejected/bad records
# we copy the columns from the main dataframe and append the Reject_Reason column
rejected_records = pd.DataFrame(columns=df.columns.tolist() + ["Reject_Reason"])

print("Initialized rejected_records accumulator")

Initialized rejected_records accumulator


### Overall insights

In [288]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   ID              100 non-null    int64  
 1   Customer_Name   100 non-null    str    
 2   Order_ID        100 non-null    str    
 3   Order_Date      100 non-null    str    
 4   Product         100 non-null    str    
 5   Category        92 non-null     str    
 6   Quantity        95 non-null     str    
 7   Price           95 non-null     str    
 8   Payment_Method  100 non-null    str    
 9   Status          100 non-null    str    
 10  Total           86 non-null     float64
dtypes: float64(1), int64(1), str(9)
memory usage: 8.7 KB


Data type converions:
- `Order_Date` to datetime
- `Quantity` to int
- `Price` to float

Inter-column relations:
- `Customer_Name` = Customer_`ID` (but this is because the data is synthetic so we shouldn't be bothered here)
- $\text{Quantity} \times \text{Price} = \text{Total}$

In [289]:
df.isna().sum()

ID                 0
Customer_Name      0
Order_ID           0
Order_Date         0
Product            0
Category           8
Quantity           5
Price              5
Payment_Method     0
Status             0
Total             14
dtype: int64

## Data Fromatting and Type Conversion

### `Order_Date` to datetime64[us]

In [290]:
df["Order_Date"].head(10)

0    11/22/2024
1      7/5/2025
2    12/23/2024
3     3/19/2025
4    10/20/2025
5    11/20/2024
6      2/2/2025
7      1/3/2025
8    10/23/2025
9      5/3/2025
Name: Order_Date, dtype: str

Does every value follow this `^\d{1,2}\/\d{1,2}\/\d{4}$` format (for example 11/22/2024)?

In [291]:
order_date_mask = df["Order_Date"].str.strip().str.match(r"^\d{1,2}\/\d{1,2}\/\d{4}$")

print("Non-matching values:", (~order_date_mask).sum())

display(df["Order_Date"][~order_date_mask])

Non-matching values: 2


Non-matching values: 2


14    Jan 5 2023
92           abc
Name: Order_Date, dtype: str

"Jan 5 2023" will automatically be converted to the majority format while "abc" will just be NaT.

In [292]:
df["Order_Date"] = pd.to_datetime(df["Order_Date"], format="%m/%d/%Y", errors="coerce")

Now every value is in YYYY-MM-DD format.

In [293]:
display(df["Order_Date"].head())

0   2024-11-22
1   2025-07-05
2   2024-12-23
3   2025-03-19
4   2025-10-20
Name: Order_Date, dtype: datetime64[us]

### `Quantity` to Int64

In [294]:
df["Quantity"].head()

0    3
1    2
2    1
3    5
4    1
Name: Quantity, dtype: str

Check if every value resembles a number `\d+`.

In [295]:
quantity_mask = df["Quantity"].str.strip().str.match(r"^\d+$")

print("Non-matching values:", (~quantity_mask).sum())

display(df["Quantity"][~quantity_mask])

Non-matching values: 8


Non-matching values: 8


6     NaN
17     -2
34     -5
64    NaN
67    NaN
92     4a
93    NaN
96    NaN
Name: Quantity, dtype: str

We convert all these 8 values to NaN.

In [296]:
# keep strings that match the mask, replace everything else with None
df["Quantity"] = df["Quantity"].where(quantity_mask, None)

# convert the entire column to numbers and give it the nullable integer type
df["Quantity"] = pd.to_numeric(df["Quantity"]).astype("Int64")

display(df["Quantity"].head())

0    3
1    2
2    1
3    5
4    1
Name: Quantity, dtype: Int64

### `Price` to Float64

In [297]:
df["Price"].head()

0        38
1       abd
2    389.05
3    233.92
4    552.51
Name: Price, dtype: str

Check for the values that resemble an actual number `^\d+\.{0,1}\d*$`.

In [298]:
price_mask = df["Price"].str.strip().str.match(r"^\d+\.{0,1}\d*$")

print("Non-matching values:", (~price_mask).sum())

display(df["Price"][~price_mask])

Non-matching values: 10


Non-matching values: 10


1              abd
10    four hundred
16             NaN
20            300$
24             NaN
30             NaN
37            -100
56             NaN
83             NaN
96             abd
Name: Price, dtype: str

Values like "four hundred", "300$", and "-100" are among the non-matching values. It's tempting whether to discard them as NaN or convert them to their most probably meant value. But since they're just 3 in number, it's better to discard and note them down along with their indices to report them accordingly.

|Index|Price|
|---|---|
|10|four hundred|
|20|300$|
|37|-100|

In [299]:
df["Price"] = df["Price"].where(price_mask, None)
df["Price"] = pd.to_numeric(df["Price"]).astype("Float64")

display(df["Price"].head())

0      38.0
1      <NA>
2    389.05
3    233.92
4    552.51
Name: Price, dtype: Float64

### Formatting strings

In [300]:
str_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()

print("String type columns:", str_cols)
display(df[str_cols].head())

String type columns: ['Customer_Name', 'Order_ID', 'Product', 'Category', 'Payment_Method', 'Status']


String type columns: ['Customer_Name', 'Order_ID', 'Product', 'Category', 'Payment_Method', 'Status']


,Customer_Name,Order_ID,Product,Category,Payment_Method,Status
0,Customer_100,ORD-41285,Blender,Home,Cash on Delivery,Shipped
1,Customer_101,ORD-35783,Smartphone,Electronics,PayPal,Processing
2,Customer_102,ORD-84355,Tennis Racket,Sports,PayPal,Delivered
3,Customer_103,ORD-57811,Science,Books,PayPal,Processing
4,Customer_104,ORD-93614,Biography,Books,Cash on Delivery,Processing


In [301]:
# Trailing whitespace handling
for col in str_cols:
    df[col] = df[col].str.strip()

In [302]:
def format_masker(df: pd.DataFrame, col_format: list[tuple[str, str]], show=False):
    for col, format in col_format:
        mask = df[col].str.match(format)

        print(f"Non-matching values for column {col} and format {format}:", (~mask).sum())
        if show:
            display(df[col][(~mask)])
            print("-"*64)

col_format_pair = [("Customer_Name", r"^Customer_\d{3}$"), ("Order_ID", r"^ORD-\d{5}$")]
format_masker(df, col_format_pair)

Non-matching values for column Customer_Name and format ^Customer_\d{3}$: 0
Non-matching values for column Order_ID and format ^ORD-\d{5}$: 0


In [303]:
unique_method_cols = ["Product", "Category", "Payment_Method", "Status"]

for col in unique_method_cols:
    df[col] = df[col].str.title()
    print(f"{col}:", sorted(df[col].dropna().unique().tolist()))
    print("")

Product: ['Basketball', 'Biography', 'Blender', 'Comics', 'Fiction', 'Football', 'Headphones', 'Jacket', 'Jeans', 'Lamp', 'Laptop', 'Microwave', 'Science', 'Shoes', 'Smartphone', 'Smartwatch', 'T-Shirt', 'Tennis Racket', 'Vacuum', 'Yoga Mat']

Category: ['Books', 'Clothing', 'Electronic', 'Electronics', 'Home', 'Sports']

Payment_Method: ['Bank Transfer', 'Cash On Delivery', 'Credit Card', 'Paypal']

Status: ['Cancelled', 'Delivered', 'Processing', 'Returned', 'Shipped']



Every unique value is a valid value. Though, we need to make "Electronic" values plural in column Category to match with "Electronics" values.

In [304]:
df["Category"] = df["Category"].str.replace(r"^Electronic$", "Electronics", regex=True)

print("Category:", sorted(df["Category"].dropna().unique().tolist()))

Category: ['Books', 'Clothing', 'Electronics', 'Home', 'Sports']


### Conclusion

In [305]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   ID              100 non-null    int64         
 1   Customer_Name   100 non-null    str           
 2   Order_ID        100 non-null    str           
 3   Order_Date      98 non-null     datetime64[us]
 4   Product         100 non-null    str           
 5   Category        92 non-null     str           
 6   Quantity        92 non-null     Int64         
 7   Price           90 non-null     Float64       
 8   Payment_Method  100 non-null    str           
 9   Status          100 non-null    str           
 10  Total           86 non-null     float64       
dtypes: Float64(1), Int64(1), datetime64[us](1), float64(1), int64(1), str(6)
memory usage: 8.9 KB


In [306]:
df.isna().sum()

ID                 0
Customer_Name      0
Order_ID           0
Order_Date         2
Product            0
Category           8
Quantity           8
Price             10
Payment_Method     0
Status             0
Total             14
dtype: int64

A table showing the NA count change after data type conversion

| Column      | Old NA  | New NA    |
| ---         | ---     | ---       |
| `Order_Date`| 0       | 2         |
| `Quantity`  | 5       | 8         |
| `Price`     | 5       | 10        |
|             | **10**  | **20**    |












## ROMI Approach

### Relation

Track the rows where $\text{Quantity} \times \text{Price} \ne \text{Total}$

In [307]:
col_quantity = df["Quantity"]
col_price = df["Price"]
col_total = df["Total"]

ne_mask = (col_quantity * col_price != col_total)

print("Rows with broken relations:", ne_mask.sum())
display(df[ne_mask])

Rows with broken relations: 9


Rows with broken relations: 9


,ID,Customer_Name,Order_ID,Order_Date,Product,Category,Quantity,Price,Payment_Method,Status,Total
7,107,Customer_107,ORD-89122,2025-01-03,Biography,Books,5,587.68,Paypal,Returned,2938.400
15,115,Customer_115,ORD-72778,2025-01-10,Fiction,Books,2,796.84,Cash On Delivery,Returned,1115.576
42,142,Customer_142,ORD-69018,2025-10-30,Shoes,Clothing,5,645.26,Credit Card,Shipped,2258.410
51,151,Customer_151,ORD-56552,2025-07-01,Comics,Books,2,457.16,Paypal,Processing,640.024
60,160,Customer_160,ORD-96037,2025-01-03,Science,Books,5,242.52,Cash On Delivery,Returned,1212.600
73,173,Customer_173,ORD-42171,2025-10-11,Lamp,Home,5,351.89,Credit Card,Shipped,1759.450
79,179,Customer_179,ORD-99328,2025-09-03,Yoga Mat,Sports,3,868.67,Paypal,Delivered,2606.010
85,185,Customer_185,ORD-57123,2025-08-14,Football,Sports,2,730.92,Cash On Delivery,Cancelled,1023.288
99,199,Customer_199,ORD-82922,2025-01-22,Blender,Home,5,372.28,Credit Card,Shipped,1861.400


If `Total` is meant to be exactly `Quantity` × `Price`, then the inequality is worth investigating. It can point to data entry mistakes, rounding issues, or values that were coerced during cleaning. If `Total` includes tax, shipping, discounts, or refunds, then the inequality may be completely normal and you should not treat it as an error.

In our case, such clarifying information is not provided. So I choose to keep them as-is.

### Outlier

This **Outlier** step is for numeric values. We check for the existence of weird and inappropriate (extreme) values.

In [308]:
display(df.describe())

,ID,Order_Date,Quantity,Price,Total
count,100.000000,98,92.0,90.0,86.000000
mean,149.500000,2025-04-23 15:55:06.122449,3.065217,625.102333,1221.132302
min,100.000000,2023-01-05 00:00:00,1.0,38.0,-20000.000000
25%,124.750000,2025-02-09 00:00:00,2.0,276.05,586.670000
50%,149.500000,2025-05-18 12:00:00,3.0,542.195,1127.180000
75%,174.250000,2025-08-01 12:00:00,5.0,740.4,2138.060000
max,199.000000,2025-11-06 00:00:00,5.0,10000.0,4722.700000
std,29.011492,NaN,1.510435,1032.449853,2639.151495


`Price`: 75% of the prices are within the range 273.0 and 737.94. Having values like 38.0 and 10,000.0 is suspicious, requiring further investigation.

`Total`: Having a minimum value of -20,000 is utterly preposterous.

#### Investigating `Price`

In [309]:
Q1_price = df["Price"].quantile(0.25)
Q3_price = df["Price"].quantile(0.75)
IQR_price = Q3_price - Q1_price

print(f"Q1 = {Q1_price}")
print(f"Q3 = {Q3_price}")
print(f"IQR_price = {IQR_price:.2f}")

lower_price_limit = 100
upper_price_limit = Q3_price + IQR_price

lower_price_mask = df["Price"] < lower_price_limit
upper_price_mask = df["Price"] > upper_price_limit

Q1 = 276.05
Q3 = 740.4
IQR_price = 464.35


In [310]:
# Checking lower limit (prices less than 100)
display(df[lower_price_mask])

,ID,Customer_Name,Order_ID,Order_Date,Product,Category,Quantity,Price,Payment_Method,Status,Total
0,100,Customer_100,ORD-41285,2024-11-22,Blender,Home,3,38.0,Cash On Delivery,Shipped,114.00
55,155,Customer_155,ORD-20182,2025-05-01,Shoes,Clothing,1,40.95,Paypal,Returned,40.95
77,177,Customer_177,ORD-64136,2025-07-28,Laptop,Electronics,1,69.48,Bank Transfer,Returned,69.48


It's hard to believe 3 blenders cost only 38.0 price units. Even a laptop costing 69.48 is ridicilous. So I dedcided to note the case down and move on. And honestly, it's really not part of my job or responsibility to indentify such meaninglessness. But if they are part of the outliers, then yes it's my job to handle and report those.

In [311]:
# excluding the shoes product
lower_price_mask = (df["Price"] < 100) & (df["Product"] != "Shoes")

In [312]:
# Checking upper limit
display(df[upper_price_mask])

,ID,Customer_Name,Order_ID,Order_Date,Product,Category,Quantity,Price,Payment_Method,Status,Total
17,117,Customer_117,ORD-72751,2025-02-12,Blender,Electronics,<NA>,10000.0,Cash On Delivery,Processing,-20000.0


A price of 10,000.0 for blender (unknown quantity), relative to the rest of the dataset, is an obvious outlier. Even the total is -20,000.0. In addition, according to other blender sales, the category should be home, not electronics. So the question is, which option should I choose:

1. I drop the record
2. I make the nonsense entries NA
3. I drop the row after saving it in another dropped row accumulater dataframe

Four entries are invalid (nonsense), which accounts for the heart/essence of the record. In other words, this record is irreparably bad data. So option 2 is out of the options. Option 1 is data loss. If the client asks, "Why does the final dataset have one less row than the raw dataset?" we have nothing to show them. That's why we go with option 3. It even is the most professional way.

In [313]:
outlier_price_rows = pd.concat([df[lower_price_mask], df[upper_price_mask]]).copy()
outlier_price_rows["Reject_Reason"] = "Outlier Price"

print("Outlier price rows:")
display(outlier_price_rows)

rejected_records = pd.concat([rejected_records, outlier_price_rows])
print("Updated rejected records:")
display(rejected_records)

df.drop(outlier_price_rows.index, inplace=True)

Outlier price rows:


,ID,Customer_Name,Order_ID,Order_Date,Product,Category,Quantity,Price,Payment_Method,Status,Total,Reject_Reason
0,100,Customer_100,ORD-41285,2024-11-22,Blender,Home,3,38.0,Cash On Delivery,Shipped,114.00,Outlier Price
77,177,Customer_177,ORD-64136,2025-07-28,Laptop,Electronics,1,69.48,Bank Transfer,Returned,69.48,Outlier Price
17,117,Customer_117,ORD-72751,2025-02-12,Blender,Electronics,<NA>,10000.0,Cash On Delivery,Processing,-20000.00,Outlier Price


Outlier price rows:


,ID,Customer_Name,Order_ID,Order_Date,Product,Category,Quantity,Price,Payment_Method,Status,Total,Reject_Reason
0,100,Customer_100,ORD-41285,2024-11-22,Blender,Home,3,38.0,Cash On Delivery,Shipped,114.00,Outlier Price
77,177,Customer_177,ORD-64136,2025-07-28,Laptop,Electronics,1,69.48,Bank Transfer,Returned,69.48,Outlier Price
17,117,Customer_117,ORD-72751,2025-02-12,Blender,Electronics,<NA>,10000.0,Cash On Delivery,Processing,-20000.00,Outlier Price


Updated rejected records:


,ID,Customer_Name,Order_ID,Order_Date,Product,Category,Quantity,Price,Payment_Method,Status,Total,Reject_Reason
0,100,Customer_100,ORD-41285,2024-11-22 00:00:00,Blender,Home,3,38.0,Cash On Delivery,Shipped,114.0,Outlier Price
77,177,Customer_177,ORD-64136,2025-07-28 00:00:00,Laptop,Electronics,1,69.48,Bank Transfer,Returned,69.48,Outlier Price
17,117,Customer_117,ORD-72751,2025-02-12 00:00:00,Blender,Electronics,<NA>,10000.0,Cash On Delivery,Processing,-20000.0,Outlier Price


#### Investigating `Total`

Our only issue with this `Total` column is the negative minimum value. The higher values are good.

Real world sales datasets may use negative totals for Returns or Refunds. But here, none of the negative total rows has a Return or Refund status. So, we proceed to investigate these values.

In [314]:
lower_total_limit = 100
lower_total_mask = df["Total"] < lower_total_limit

# Checking lower limit (prices less than 100)
display(df[lower_total_mask])

,ID,Customer_Name,Order_ID,Order_Date,Product,Category,Quantity,Price,Payment_Method,Status,Total
34,134,Customer_134,ORD-16585,2025-10-04,T-Shirt,Clothing,<NA>,591.53,Paypal,Shipped,-2957.65
37,137,Customer_137,ORD-91254,2025-02-23,Vacuum,Home,3,<NA>,Cash On Delivery,Delivered,-300.00
55,155,Customer_155,ORD-20182,2025-05-01,Shoes,Clothing,1,40.95,Paypal,Returned,40.95


The third row is fine, but not the first two; those with negative total values. So, once again, we know dropping records from such a sales dataset is crucially alarming. That's why we save them in the `rejected_rows` dataframe before dropping them.

In [315]:
outlier_total_rows = df[lower_total_mask][df[lower_total_mask]["Total"] < 0].copy()
outlier_total_rows["Reject_Reason"] = "Negative Total"

print("Outlier total rows:")
display(outlier_total_rows)

rejected_records = pd.concat([rejected_records, outlier_total_rows])
print("Updated rejected records")
display(rejected_records)

df.drop(outlier_total_rows.index, inplace=True)

Outlier total rows:


Outlier total rows:


,ID,Customer_Name,Order_ID,Order_Date,Product,Category,Quantity,Price,Payment_Method,Status,Total,Reject_Reason
34,134,Customer_134,ORD-16585,2025-10-04,T-Shirt,Clothing,<NA>,591.53,Paypal,Shipped,-2957.65,Negative Total
37,137,Customer_137,ORD-91254,2025-02-23,Vacuum,Home,3,<NA>,Cash On Delivery,Delivered,-300.00,Negative Total


Outlier total rows:


,ID,Customer_Name,Order_ID,Order_Date,Product,Category,Quantity,Price,Payment_Method,Status,Total,Reject_Reason
34,134,Customer_134,ORD-16585,2025-10-04,T-Shirt,Clothing,<NA>,591.53,Paypal,Shipped,-2957.65,Negative Total
37,137,Customer_137,ORD-91254,2025-02-23,Vacuum,Home,3,<NA>,Cash On Delivery,Delivered,-300.00,Negative Total


Updated rejected records


,ID,Customer_Name,Order_ID,Order_Date,Product,Category,Quantity,Price,Payment_Method,Status,Total,Reject_Reason
0,100,Customer_100,ORD-41285,2024-11-22 00:00:00,Blender,Home,3,38.0,Cash On Delivery,Shipped,114.0,Outlier Price
77,177,Customer_177,ORD-64136,2025-07-28 00:00:00,Laptop,Electronics,1,69.48,Bank Transfer,Returned,69.48,Outlier Price
17,117,Customer_117,ORD-72751,2025-02-12 00:00:00,Blender,Electronics,<NA>,10000.0,Cash On Delivery,Processing,-20000.0,Outlier Price
34,134,Customer_134,ORD-16585,2025-10-04 00:00:00,T-Shirt,Clothing,<NA>,591.53,Paypal,Shipped,-2957.65,Negative Total
37,137,Customer_137,ORD-91254,2025-02-23 00:00:00,Vacuum,Home,3,<NA>,Cash On Delivery,Delivered,-300.0,Negative Total


### Mismatch

This **Mismatch** step is for strings (categorical values). In our case, we should check if, for example, every blender is categorized as a home utility and not electronics or something else.

For this specific dataset, we're going to match `Product` and `Category` columns.

In [316]:
# count how many unique categories each product has
category_counts = df.groupby("Product")["Category"].nunique()
display(category_counts)
print("-"*64)

problem_products = category_counts[category_counts > 1].index

if len(problem_products) > 0:
    print("Mismatched products found:")
    mismatched_rows = df[df["Product"].isin(problem_products)][["Product", "Category"]].drop_duplicates().dropna()
    display(mismatched_rows.sort_values(by="Product"))
else:
    print("No mismatches found. Every product belongs to exactly one category.")

Product
Basketball       2
Biography        1
Blender          1
Comics           1
Fiction          1
Football         1
Headphones       1
Jacket           1
Jeans            1
Lamp             2
Laptop           1
Microwave        2
Science          1
Shoes            1
Smartphone       1
Smartwatch       1
T-Shirt          2
Tennis Racket    1
Vacuum           2
Yoga Mat         2
Name: Category, dtype: int64

Product
Basketball       2
Biography        1
Blender          1
Comics           1
Fiction          1
Football         1
Headphones       1
Jacket           1
Jeans            1
Lamp             2
Laptop           1
Microwave        2
Science          1
Shoes            1
Smartphone       1
Smartwatch       1
T-Shirt          2
Tennis Racket    1
Vacuum           2
Yoga Mat         2
Name: Category, dtype: int64

----------------------------------------------------------------
Mismatched products found:


,Product,Category
10,Basketball,Sports
46,Basketball,Electronics
28,Lamp,Electronics
31,Lamp,Home
16,Microwave,Home
74,Microwave,Electronics
43,T-Shirt,Clothing
66,T-Shirt,Electronics
23,Vacuum,Electronics
90,Vacuum,Home


In [317]:
product_category_map = {
    "Basketball": "Sports",
    "Lamp": "Home",
    "Microwave": "Home",
    "T-Shirt": "Clothing",
    "Vacuum": "Home",
    "Yoga Mat": "Sports"
}

df["Category"] = df["Product"].map(product_category_map).fillna(df["Category"])

# Optional: verify the fix worked by rerunning the check
category_counts = df.groupby("Product")["Category"].nunique()
problem_products = category_counts[category_counts > 1].index
print("Remaining mismatch products:", len(problem_products))

Remaining mismatch products: 0


### Interpolation

This **Interpolation** step is for filling in empty (NA) values. However, there is no need of interpolation for this dataset because, come on, this is a sales dataset; You can't just plug in best guess prices. As a result, empty values are left as-is. But make sure to write this decision down in the final report.

## Final Check

In [318]:
df.info()

<class 'pandas.DataFrame'>
Index: 95 entries, 1 to 99
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   ID              95 non-null     int64         
 1   Customer_Name   95 non-null     str           
 2   Order_ID        95 non-null     str           
 3   Order_Date      93 non-null     datetime64[us]
 4   Product         95 non-null     str           
 5   Category        89 non-null     str           
 6   Quantity        89 non-null     Int64         
 7   Price           86 non-null     Float64       
 8   Payment_Method  95 non-null     str           
 9   Status          95 non-null     str           
 10  Total           81 non-null     float64       
dtypes: Float64(1), Int64(1), datetime64[us](1), float64(1), int64(1), str(6)
memory usage: 9.1 KB


In [319]:
df.describe()

,ID,Order_Date,Quantity,Price,Total
count,95.000000,93,89.0,86.0,81.000000
mean,150.368421,2025-04-23 22:11:36.774193,3.089888,529.769767,1581.377136
min,101.000000,2023-01-05 00:00:00,1.0,40.95,40.950000
25%,125.500000,2025-02-09 00:00:00,2.0,293.18,696.710000
50%,151.000000,2025-05-19 00:00:00,3.0,542.195,1212.600000
75%,174.500000,2025-08-03 00:00:00,5.0,740.4,2258.410000
max,199.000000,2025-11-06 00:00:00,5.0,978.63,4722.700000
std,28.915524,NaN,1.519872,255.411039,1166.505478


In [320]:
df.isna().sum()

ID                 0
Customer_Name      0
Order_ID           0
Order_Date         2
Product            0
Category           6
Quantity           6
Price              9
Payment_Method     0
Status             0
Total             14
dtype: int64

In [321]:
na_rows_mask = df.isna().any(axis=1)

print(f"There are a total of {na_rows_mask.sum()} rows with at least 1 empty (NA) value.")

rows_with_na = df[na_rows_mask]
display(rows_with_na.head())

There are a total of 21 rows with at least 1 empty (NA) value.


,ID,Customer_Name,Order_ID,Order_Date,Product,Category,Quantity,Price,Payment_Method,Status,Total
1,101,Customer_101,ORD-35783,2025-07-05,Smartphone,Electronics,2,<NA>,Paypal,Processing,NaN
6,106,Customer_106,ORD-25885,2025-02-02,Blender,Home,<NA>,978.63,Bank Transfer,Processing,NaN
10,110,Customer_110,ORD-61020,2025-09-26,Basketball,Sports,5,<NA>,Cash On Delivery,Cancelled,NaN
14,114,Customer_114,ORD-77417,NaT,Smartphone,Electronics,2,413.13,Cash On Delivery,Shipped,826.26
16,116,Customer_116,ORD-63660,2025-10-30,Microwave,Home,4,<NA>,Cash On Delivery,Processing,NaN


## Export

In [322]:
df.to_csv("data/clean.csv", date_format="%Y-%m-%d", index=False)
print("Saved successfully!")

Saved successfully!
